<font color="#0D9DF0"><h3><b>ANÁLISE DO DESENVOLVIMENTO HUMANO E ECONÔMICO GLOBAL</b></h3></font>

**Base de dados:** Gapminder  
**Grupo:** [preencher nomes]  
**Etapa atual:** Planejamento

Objetivo do trabalho
Responder as seguintes questoes tendo com dado base inicial o gapminder disponibilizado pelo professor

1- Países que gastam mais em saúde per capita têm maior expectativa de vida, mesmo quando o PIB per capita é parecido?

2- A mortalidade infantil explica melhor as diferenças de expectativa de    vida entre países ricos e pobres do que o PIB per capita?

3 - Países com mais veículos por habitante têm melhor longevidade?

4 - A taxa de obesidade cresce junto com o PIB per capita?

5 - Países com mais médicos por mil habitantes têm maior expectativa de
vida, mesmo descontando o efeito do PIB per capita?





Etapas do notebook
1. Coleta e exploração inicial
2. Limpeza e preparação
3. Análise exploratória
4. Funções de visualização
5. Interpretação das análises
6. Síntese e conclusão

#### Importação das bibliotecas

In [151]:
# Importar as bibliotecas que serão utilizadas pelo grupo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import csv
from charset_normalizer import from_bytes #Para detectar encoding e separador dos arquivos automaticamente
import urllib.request as ur   #Para aceitar base de dados que venham de URL

---

## 1. Coleta de Dados
**Etapa atual:  COLETA**

O primeiro passo é carregar a base Gapminder e registrar sua origem. Confiram se o arquivo foi disponibilizado junto com o trabalho e anotem a fonte utilizada.

**Links bases de dados**

In [152]:
# Substitua a URL abaixo pela URL 'raw' do seu arquivo no GitHub
url_github = 'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/gapminder_full.csv'


**Função para descobrir o encoding e o separador do arquivo**


In [180]:
def configuraçoes_csv(caminho_arquivo):
    """
    Descobre o encoding e o separador de qualquer CSV de forma rápida e certeira.
    Aceita tanto arquivos locais quanto links da internet (ex: GitHub).
    Retorna: (encoding, separador).
    """
    caminho_arquivo = str(caminho_arquivo)  # Garante que o caminho seja uma string
    # 1. Verifica se é um link da internet ou arquivo local
    if caminho_arquivo.startswith('http://') or caminho_arquivo.startswith('https://'):   # Verifica se começa com um dos termos
        # Se for link, faz uma requisição HTTP e lê os primeiros 20.000 bytes
        req = ur.Request(
            caminho_arquivo,
            headers={'User-Agent': 'Mozilla/5.0'} # Simula um cabeçalho real na requisição como se fosse um usuário acessando pelo Mozila
        )
        with ur.urlopen(req) as f:   # Abre a conexão com o servidor através da requisição anterior criada
            amostra_binaria = f.read(20000)   #Lê e armazena os primeiros 20000 bytes em amostra_binária
                                              #NOTA: não usamos open(caminho_arquivo, "rb") pq na internet tudo já está em byte
    else:
        # Se for arquivo local, abre normalmente
        with open(caminho_arquivo, 'rb') as f:    #Aqui usamos o 'rb' pois arquivos locais não costuman estar em binário pois nós não conseguiriamos ler.
            amostra_binaria = f.read(20000)

    # 2. Descobre o encoding com quase 100% de precisão
    resultado = from_bytes(amostra_binaria).best()  # Analisa a amostra binária
    encoding = resultado.encoding   # Armazena o nome do encoding detectado
    texto_amostra = str(resultado)    # Converte em texto legível novamente

    # 3. Descobre o separador (.csv, .txt, etc)
    dialeto = csv.Sniffer().sniff(texto_amostra)    #"Fareja" em nossa amostra qual o separador mais aparece e
    separador = dialeto.delimiter

    return caminho_arquivo,encoding, separador

In [195]:
cam,enc,sep = configuraçoes_csv(url_github) #Guarda cada dado da tupla em uma variável 'enc' e 'rep'


**Função para criar o Data Frame**

In [197]:
def cria_dataframe(caminho_arquivo,encoding,separador):
    df=pd.read_csv(caminho_arquivo, encoding=encoding, sep=separador)
    return df


In [198]:
criar_dataframe(cam,enc,sep)

,country,year,pop,continent,lifeExp,gdpPercap
0,Afghanistan,1952,8425333.0,Asia,28.801,779.445314
1,Afghanistan,1957,9240934.0,Asia,30.332,820.853030
2,Afghanistan,1962,10267083.0,Asia,31.997,853.100710
3,Afghanistan,1967,11537966.0,Asia,34.020,836.197138
4,Afghanistan,1972,13079460.0,Asia,36.088,739.981106
...,...,...,...,...,...,...
1731,Zimbabwe,1987,9216418.0,Africa,62.351,706.157306
1732,Zimbabwe,1992,10704340.0,Africa,60.377,693.420786
1733,Zimbabwe,1997,11404948.0,Africa,46.809,792.449960
1734,Zimbabwe,2002,11926563.0,Africa,39.989,672.038623


In [192]:
def carregar_multiplos_dataframes(lista_caminhos):
    """
    Recebe uma lista de URLs ou caminhos locais e retorna uma lista de DataFrames carregados
    automaticamente usando as configurações detectadas para cada um deles.

    **Exemplo de entrada:**
    lista caminhos= [url1, url2, url3, ...] ou [caminho1, caminho2, caminho3, ...]

    **Exemplo de saída:**
    dataframes= [df1, df2, df3, ...]
    """
    dataframes = []   #cria uma lista vazia para nosso append ir add oq precisa
    for caminho in lista_caminhos:
        # 1. Detecta as configurações do arquivo atual
        config = configuraçoes_csv(caminho)
        # 2. Cria o DataFrame correspondente
        df = cria_dataframe(*config)#garante que cada termo da tupla seja descompactado e utilizado na função
        # 3. Adiciona o DataFrame à lista
        dataframes.append(df)
    return dataframes

# Exemplo de uso prático:
# lista_de_fontes = [url_github]  # Você pode colocar quantos links ou caminhos quiser aqui
# meus_dfs = carregar_multiplos_dataframes(lista_de_fontes)
# meus_dfs[0].head()  # Acessa o primeiro DataFrame gerado

In [202]:
carregar_multiplos_dataframes({url_github})[0] #Escolha qual dataframe quer ver da nossa lista pelo indice

,country,year,pop,continent,lifeExp,gdpPercap
0,Afghanistan,1952,8425333.0,Asia,28.801,779.445314
1,Afghanistan,1957,9240934.0,Asia,30.332,820.853030
2,Afghanistan,1962,10267083.0,Asia,31.997,853.100710
3,Afghanistan,1967,11537966.0,Asia,34.020,836.197138
4,Afghanistan,1972,13079460.0,Asia,36.088,739.981106
...,...,...,...,...,...,...
1731,Zimbabwe,1987,9216418.0,Africa,62.351,706.157306
1732,Zimbabwe,1992,10704340.0,Africa,60.377,693.420786
1733,Zimbabwe,1997,11404948.0,Africa,46.809,792.449960
1734,Zimbabwe,2002,11926563.0,Africa,39.989,672.038623


## 2. Exploração Inicial dos Dados
**Etapa atual:  EXPLORAÇÃO INICIAL**

Antes de tratar os dados, verificar a estrutura do DataFrame e entender os itens em cada colunas.

In [ ]:
# Visualizar informações gerais do DataFrame

In [ ]:
# Consultar estatísticas descritivas

In [ ]:
# Verificar os países, continentes e anos presentes na base

**Anotações do grupo**
- Quais colunas estão disponíveis?
- Qual é o período coberto pelos dados?
- Há algo que precisa ser verificado antes da limpeza?

## 3. Limpeza de Dados
**Etapa atual:  TRATAMENTO**

A atividade solicita excluir os dados ausentes e eliminar registros duplicados. Antes e depois da limpeza, registrem a quantidade de linhas e as decisões tomadas.

In [ ]:
# Verificar a quantidade de valores ausentes por coluna

In [ ]:
# Verificar a quantidade de registros duplicados

In [ ]:
# Remover os valores ausentes conforme solicitado no enunciado

In [ ]:
# Eliminar registros duplicados

In [ ]:
# Conferir a base após a limpeza

**Decisões de tratamento**
- Quantos valores ausentes foram encontrados?
- Quantas linhas duplicadas foram removidas?
- Quantos registros restaram?
- Foi necessário ajustar algum tipo de dado? Justifique.

## 4. Transformação e Preparação dos Dados
**Etapa atual:  PREPARAÇÃO**

Avaliem se é necessário ajustar tipos de dados, nomes de colunas ou criar agrupamentos para facilitar as análises. Registrem apenas as transformações que realmente forem utilizadas.

In [ ]:
# Realizar aqui as transformações necessárias, se houver

## 5. Análise Exploratória de Dados (AED)
**Etapa atual: 🟩 ANÁLISE**

Escolham pelo menos **15 perguntas** para investigar. Para cada análise, incluam código, tabela ou gráfico e uma interpretação escrita pelo grupo. As perguntas abaixo são sugestões; vocês podem adaptar ou substituir por outras relacionadas ao objetivo do trabalho.

### Análise 1 — Evolução da expectativa de vida
**Pergunta:** Como a expectativa de vida global mudou entre 1952 e 2007?

**Visualização possível:** Gráfico de linhas ao longo dos anos.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [ ]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 2 — Evolução do PIB per capita
**Pergunta:** Como o PIB per capita se modificou ao longo do período analisado?

**Visualização possível:** Gráfico de linhas ou comparação entre anos.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [157]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 3 — Crescimento populacional
**Pergunta:** Como a população total representada na base mudou ao longo do tempo?

**Visualização possível:** Tabela agregada ou gráfico de linhas.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [158]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 4 — Comparação entre continentes
**Pergunta:** Quais diferenças de expectativa de vida podem ser observadas entre continentes?

**Visualização possível:** Gráfico de linhas ou barras por continente.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [159]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 5 — PIB por continente
**Pergunta:** Como o PIB per capita se distribui entre os continentes em um ano escolhido?

**Visualização possível:** Gráfico de barras ou boxplot.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [160]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 6 — Expectativa de vida e PIB
**Pergunta:** Existe uma relação visual entre PIB per capita e expectativa de vida?

**Visualização possível:** Gráfico de dispersão.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [161]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 7 — Países com maior expectativa de vida
**Pergunta:** Quais países apresentam os maiores valores de expectativa de vida em um ano escolhido?

**Visualização possível:** Tabela de ranking ou gráfico de barras.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [162]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 8 — Países com menor expectativa de vida
**Pergunta:** Quais países apresentam os menores valores de expectativa de vida em um ano escolhido?

**Visualização possível:** Tabela de ranking ou gráfico de barras.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [163]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 9 — Maiores PIBs per capita
**Pergunta:** Quais países têm os maiores PIBs per capita em um ano escolhido?

**Visualização possível:** Tabela de ranking ou gráfico de barras.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [164]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 10 — Menores PIBs per capita
**Pergunta:** Quais países têm os menores PIBs per capita em um ano escolhido?

**Visualização possível:** Tabela de ranking ou gráfico de barras.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [165]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 11 — Mudanças em um país
**Pergunta:** Como a expectativa de vida e o PIB per capita de um país escolhido mudaram ao longo do tempo?

**Visualização possível:** Gráfico de linhas com evolução temporal.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [166]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 12 — Disparidade entre países
**Pergunta:** A diferença entre países com maior e menor expectativa de vida mudou ao longo do período?

**Visualização possível:** Comparação de valores por ano.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [167]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 13 — Distribuição da expectativa de vida
**Pergunta:** Como os valores de expectativa de vida estão distribuídos em um ano escolhido?

**Visualização possível:** Histograma ou boxplot.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [168]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 14 — Distribuição do PIB per capita
**Pergunta:** Como os valores de PIB per capita estão distribuídos em um ano escolhido?

**Visualização possível:** Histograma ou boxplot.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [169]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 15 — População e expectativa de vida
**Pergunta:** É possível observar algum padrão entre tamanho da população e expectativa de vida?

**Visualização possível:** Gráfico de dispersão.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [170]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 16 — Variação entre continentes
**Pergunta:** Qual continente apresentou maior mudança na expectativa de vida entre dois anos escolhidos?

**Visualização possível:** Tabela comparativa ou gráfico de barras.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [171]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 17 — Variação do PIB por continente
**Pergunta:** Como o PIB per capita médio de cada continente variou entre dois anos?

**Visualização possível:** Gráfico de barras agrupadas ou linhas.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [172]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 18 — Comparação de anos
**Pergunta:** Quais diferenças gerais aparecem ao comparar o primeiro e o último ano da base?

**Visualização possível:** Tabela comparativa e gráficos.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [173]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 19 — Valores extremos
**Pergunta:** Existem países com valores muito diferentes dos demais em expectativa de vida ou PIB per capita?

**Visualização possível:** Boxplot e inspeção dos registros.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [174]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 20 — Relação entre os três indicadores
**Pergunta:** Como expectativa de vida, PIB per capita e população se comportam em conjunto?

**Visualização possível:** Gráfico de dispersão; tamanho dos pontos pode representar população.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [175]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

## 6. Funções de Visualização e Análise Gráfica
**Etapa atual: 🟩 ORGANIZAÇÃO DO CÓDIGO**

Se houver gráficos repetidos, criem uma função simples para reaproveitar o código. Esta etapa é recomendada no enunciado, mas mantenham a solução compatível com o que foi visto em aula.

In [176]:
# Definir aqui uma função de visualização, se necessário

In [177]:
# Testar a função criada em uma das análises

## 7. Interpretação das Visualizações
**Etapa atual: 🟪 INTERPRETAÇÃO**

Reúnam as interpretações das análises escolhidas e expliquem:
- O que o gráfico ou a tabela mostra?
- Que comparação ou tendência chamou atenção?
- A pergunta inicial foi respondida?
- Existe alguma limitação que precisa ser considerada?

## 8. Síntese dos Principais Insights
**Etapa atual: 🟪 SÍNTESE**

Registrem os principais achados com base nos resultados produzidos pelo grupo.

1. **Insight 1:** [preencher com evidência da análise]
2. **Insight 2:** [preencher com evidência da análise]
3. **Insight 3:** [preencher com evidência da análise]

**Possíveis aplicações práticas:** que relação esses resultados podem ter com políticas públicas, saúde, condições econômicas ou desigualdades entre países?

## 9. Conclusão e Reflexão
**Etapa atual: 🟪 CONCLUSÃO**

- Quais foram os principais aprendizados?
- Quais foram as maiores dificuldades na coleta, limpeza ou análise?
- Quais limitações da base devem ser consideradas?
- O objetivo inicial foi atendido? Justifiquem com os resultados encontrados.

## 10. Referências
**Etapa atual: ⬜ FINALIZAÇÃO**

Registrem a fonte do dataset e quaisquer outras fontes utilizadas.

## Checklist antes da entrega
- [ ] As seis partes recomendadas no enunciado foram contempladas.
- [ ] Os valores ausentes foram removidos.
- [ ] Os registros duplicados foram eliminados.
- [ ] Foram realizadas pelo menos 15 análises.
- [ ] Cada visualização tem título, identificação dos eixos e interpretação.
- [ ] As decisões e observações estão documentadas em Markdown.
- [ ] O notebook foi executado do início ao fim e revisado.
- [ ] As análises e interpretações refletem o trabalho autoral do grupo.

**Atenção:** o enunciado proíbe o uso de IA generativa para geração automática de código ou análise. Utilizem este arquivo apenas como estrutura de organização e desenvolvam as soluções e interpretações por conta própria.